# commit-model — QLoRA fine-tuning on Kaggle

Fine-tunes `Qwen2.5-Coder-3B-Instruct` with DoRA (4-bit QLoRA) to generate
Conventional Commit messages from git diffs.

This notebook is **self-contained** — it does not clone or install anything
from the GitHub repo. Everything it needs is pasted directly into the cells
below. The source of truth for this logic still lives in the repo
(`scripts/train_lora.py`, `configs/lora_config_kaggle.yaml`,
`src/commit_model/prompts.py`) — if you change training behavior there,
mirror the change here too.

**Before running:**
1. Notebook settings (right sidebar) → **Accelerator: GPU T4 x2** (or T4 x1)
   — **not P100**. Kaggle's currently pre-installed PyTorch build only
   supports compute capability sm_70+ (Volta/Turing/Ampere/Hopper); the P100
   is Pascal (sm_60) and will fail with a `CUDA capability sm_60 is not
   compatible` error. T4 (sm_75) has the same 16 GB VRAM this config is
   sized for. Also set **Internet: On**.
2. **Add Input** (top right) → attach the Kaggle Dataset containing your
   `train.jsonl` / `valid.jsonl` / `test.jsonl` (produced locally by
   `scripts/prepare_data.py`, uploaded via kaggle.com → Datasets → New Dataset).
3. Edit `DATASET_SLUG` in the next cell to match the dataset you attached.


In [ ]:
# Kaggle ships torch pre-installed with CUDA already wired up — leave it alone.
# Only install the extra libraries this project needs.
!pip install -q -U transformers>=4.47.0 peft>=0.12.0 bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 datasets>=2.20.0 safetensors pyyaml


In [ ]:
import glob
import os

# --- Point this at your attached dataset ---
# The folder name under /kaggle/input/ matches the dataset slug you chose
# when creating it on kaggle.com (Datasets -> New Dataset).
DATASET_SLUG = "YOUR_KAGGLE_USERNAME/commit-model-data"  # <-- edit this
DATA_DIR = f"/kaggle/input/{DATASET_SLUG.split('/')[-1]}"

# If train.jsonl isn't directly at DATA_DIR, search the rest of /kaggle/input
# for it -- this handles the common case where you uploaded the whole
# data/processed/ *folder* (Kaggle then nests it one level deeper, e.g.
# /kaggle/input/commit-model-data/processed/train.jsonl) instead of the
# three loose .jsonl files.
if not os.path.exists(os.path.join(DATA_DIR, "train.jsonl")):
    matches = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find train.jsonl under {DATA_DIR} or anywhere in /kaggle/input. "
            "Did you attach the right dataset via Add Input, and does DATASET_SLUG match it?"
        )
    found_dir = os.path.dirname(matches[0])
    print(f"[!] train.jsonl not found at {DATA_DIR} -- using {found_dir} instead")
    DATA_DIR = found_dir

ADAPTER_DIR = "/kaggle/working/adapters"

# Set this to a checkpoint path (e.g. an attached previous-run Output
# containing "checkpoint-450") if you're resuming a run that didn't finish
# in one Kaggle session. Leave as None for a fresh run.
RESUME_FROM = None

config = {
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "data": DATA_DIR,
    "adapter_path": ADAPTER_DIR,
    "lora_rank": 8,
    "lora_alpha": 160,        # effective scale = alpha / rank = 20.0
    "lora_dropout": 0.0,
    "use_dora": True,
    "num_layers": 24,         # LoRA on the final 24 of Qwen2.5-Coder-3B's transformer layers
    "learning_rate": 1.0e-4,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,   # effective batch = 2 * 4 = 8
    "max_steps": 3000,
    "max_seq_length": 2048,
    "eval_steps": 50,
    "val_batches": 25,
    "save_steps": 50,
    "save_total_limit": 3,    # keep only the last 3 checkpoints -- disk fills fast otherwise
    "seed": 42,
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_use_double_quant": True,
    "optimizer": "paged_adamw_8bit",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "report_to": "none",
}
print(f"Data:    {DATA_DIR}")
print(f"Adapter: {ADAPTER_DIR}")


## Prompt format

Mirrors `src/commit_model/prompts.py` exactly. Keep both in sync — this is
the one thing that must never silently drift between training and the
local inference scripts (`infer.py`, `chat.py`, `evaluate.py`).


In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a commit message generator. Given a git diff, write a "
    "Conventional Commit message. Start with a subject line: "
    "type(scope): description (scope optional). Valid types are: feat, fix, "
    "chore, refactor, docs, test, style, perf, build, ci. You may add up to "
    "2 short body lines after a blank line if the change needs brief context. "
    "Do not include footers, issue references, URLs, or any text outside the "
    "commit message."
)
MAX_DIFF_CHARS = 12_000


def truncate_diff(diff, max_chars=MAX_DIFF_CHARS):
    if len(diff) <= max_chars:
        return diff
    return diff[:max_chars] + "\n... [diff truncated]"


def build_user_message(diff):
    return f"Diff:\n```diff\n{truncate_diff(diff)}\n```"


def build_inference_messages(diff):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(diff)},
    ]


In [ ]:
# Verify CUDA and the attached dataset are visible before starting the long run
import torch

assert torch.cuda.is_available(), "GPU not available -- enable GPU in notebook settings"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")

# Kaggle's pre-installed PyTorch build only supports sm_70+ (Volta and
# newer). The P100 (sm_60) will otherwise fail deep inside training with a
# confusing "CUDA capability sm_60 is not compatible" warning wall -- catch
# it here instead, with an actionable fix.
cap = torch.cuda.get_device_capability(0)
if cap < (7, 0):
    raise RuntimeError(
        f"{gpu} has compute capability {cap[0]}.{cap[1]}, but this notebook's "
        "PyTorch build requires 7.0+ (T4, V100, A100, ...). "
        "Go to Notebook settings (right sidebar) -> Accelerator, and switch "
        "to 'GPU T4 x2' or 'GPU T4 x1' instead of P100."
    )

for split in ("train.jsonl", "valid.jsonl"):
    p = os.path.join(DATA_DIR, split)
    assert os.path.exists(p), f"Missing: {p} -- did you attach the right dataset via Add Input?"
    with open(p) as fh:
        n = sum(1 for _ in fh)
    print(f"{split}: {n:,} examples")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model_id = config["model"]
print(f"[*] Loading tokenizer: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"

print(f"[*] Loading model in 4-bit QLoRA: {model_id}")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=config["load_in_4bit"],
    bnb_4bit_quant_type=config["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, config["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=config["bnb_4bit_use_double_quant"],
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Enable gradient checkpointing manually so we can pass use_reentrant=False.
# Essential for DoRA, which otherwise exceeds 16 GB VRAM on a T4/P100.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

total_layers = model.config.num_hidden_layers
num_lora_layers = config["num_layers"]
first_layer = total_layers - num_lora_layers

lora_config = LoraConfig(
    r=config["lora_rank"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    use_dora=config["use_dora"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    layers_to_transform=list(range(first_layer, total_layers)),
    layers_pattern="layers",
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from datasets import load_dataset

max_seq_length = config["max_seq_length"]

raw = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/valid.jsonl",
    },
)


def tokenize_and_mask(example):
    # Full conversation: system + user + assistant
    full_text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
    )
    # Prompt only: system + user (with generation prompt, to find the boundary)
    prompt_messages = [m for m in example["messages"] if m["role"] != "assistant"]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True,
    )

    full_ids = tokenizer(
        full_text, truncation=True, max_length=max_seq_length, add_special_tokens=False,
    )["input_ids"]
    prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
    prompt_len = min(prompt_len, len(full_ids))

    # Loss is only computed on the assistant's completion tokens
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


train_ds = raw["train"].map(tokenize_and_mask, remove_columns=["messages"], desc="Tokenizing train")

max_eval = config["val_batches"] * config["batch_size"]
eval_ds = raw["validation"].map(tokenize_and_mask, remove_columns=["messages"], desc="Tokenizing eval")
eval_ds = eval_ds.select(range(min(max_eval, len(eval_ds))))

print(f"Training examples: {len(train_ds):,}  |  Eval examples: {len(eval_ds):,}")


In [ ]:
from torch.nn.utils.rnn import pad_sequence

pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id


def collate_fn(examples):
    input_ids = pad_sequence(
        [torch.tensor(e["input_ids"]) for e in examples], batch_first=True, padding_value=pad_token_id,
    )
    attention_mask = pad_sequence(
        [torch.tensor(e["attention_mask"]) for e in examples], batch_first=True, padding_value=0,
    )
    labels = pad_sequence(
        [torch.tensor(e["labels"]) for e in examples], batch_first=True, padding_value=-100,
    )
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=config["adapter_path"],
    max_steps=config["max_steps"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],  # keep eval batch == train batch, or it OOMs on a 16 GB GPU
    learning_rate=config["learning_rate"],
    eval_strategy="steps",
    eval_steps=config["eval_steps"],
    save_strategy="steps",
    save_steps=config["save_steps"],
    save_total_limit=config["save_total_limit"],
    logging_steps=config["logging_steps"],
    bf16=True,
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    optim=config["optimizer"],
    warmup_steps=int(config["warmup_ratio"] * config["max_steps"]),
    report_to=config["report_to"],
    seed=config["seed"],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate_fn,
)

print(f"[*] Starting training ({config['max_steps']} steps)")
trainer.train(resume_from_checkpoint=RESUME_FROM)

print(f"[*] Saving final adapter to {config['adapter_path']}")
trainer.save_model()
print("[*] Training complete.")


## Quick smoke test

Feed the freshly-trained adapter one sample diff and eyeball the output.


In [ ]:
# `model` here is the same PeftModel object that was just trained above --
# no need to reload it from disk (it already holds the trained LoRA weights).
SAMPLE_DIFF = '''\
diff --git a/src/auth.py b/src/auth.py
index 83db48f..b3c1e5a 100644
--- a/src/auth.py
+++ b/src/auth.py
@@ -12,6 +12,10 @@ def login(username, password):
     if not user:
         raise AuthError("invalid credentials")
+
+    if user.is_locked:
+        raise AuthError("account locked, contact support")
+
     return generate_token(user)
'''

model.eval()

messages = build_inference_messages(SAMPLE_DIFF)
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]

with torch.inference_mode():
    output_ids = model.generate(
        **inputs, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True).strip())


## Getting the trained adapter off Kaggle

1. Go to the **Output** tab on the right side of this notebook (after
   **Save Version → Save & Run All**).
2. Open `adapters/` and download the files (or download the whole output as
   a zip).
3. Copy them into `models/adapters/` in your local repo clone to use with
   `scripts/infer.py`, `scripts/chat.py`, and `scripts/evaluate.py`.

## If a run doesn't finish in one session

Kaggle GPU sessions are capped (and there's a weekly GPU-hour quota). If
training gets cut off:
1. Save this notebook version anyway — its Output tab still has the latest
   `checkpoint-N/` folders under `adapters/`.
2. Start a new session, attach that previous Output as an input (Kaggle
   lets you add a notebook's own Output as a data source), and set
   `RESUME_FROM` above to the mounted `checkpoint-N` path.
3. Re-run all cells — `trainer.train(resume_from_checkpoint=RESUME_FROM)`
   picks up the optimizer state and step counter from there.
